# Single-video inference & probability plots

Этот ноутбук нужен для:
- взять **preset**
- собрать модель
- загрузить **checkpoint/weights**
- прогнать **одно видео**
- построить график **probability / pred / true**
- сохранить per-frame CSV

Подходит под твой текущий пайплайн ABAW/DVD и использует ту же логику `infer_video_per_frame`.


In [ ]:
# ===== 0) Imports =====
import os, json, math, random
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms.functional as TF

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

try:
    import av  # PyAV
except Exception:
    av = None

os.environ["CUDA_VISIBLE_DEVICES"] = "1"


In [ ]:
# ===== 1) Main config =====
CFG = dict(
    root="/home/HDD6TB/datasets/emotions/ABAW/ABAW_9/DVD",
    train_frames="Training/frames",
    val_frames="Validation/frames",
    train_anns="Training/annotations",
    val_anns="Validation/annotations",
    video_ext=".mp4",
    img_size=224,
    clip_len=32,
    frame_step=2,
    backend="frames",     # "frames" | "pyav" | "torchvision"
    seed=67,
    backbone=None,
)

# ---- choose preset/checkpoint here ----
PRESET_PATH = Path("improved_presets.json")
PRESET_NAME = "convnext_tiny_clip32"   # change if needed

# checkpoint can be either:
# 1) direct path to best.pt / epoch_x.pt
# 2) run_dir, then CKPT_PATH=None and RUN_DIR will be used
RUN_DIR = None
CKPT_PATH = "runs/convnext_tiny_clip32 0.78/best.pt"

# ===== Choose only split + video number here =====
# Examples:
#   SPLIT = "Validation"; VIDEO_NUM = "0007"
#   SPLIT = "Training";   VIDEO_NUM = "0227"
SPLIT = "Validation"       # "Validation" | "Training"
VIDEO_NUM = "1312"         # string with leading zeros preferred; int also works

def _normalize_video_id(x):
    if isinstance(x, int):
        return f"{x:04d}"
    s = str(x).strip()
    if s.endswith(".mp4"):
        s = Path(s).stem
    return s

VIDEO_ID = _normalize_video_id(VIDEO_NUM)

# These are auto-built from SPLIT and VIDEO_ID. Usually you do not need to edit them.
ROOT = Path(CFG["root"])
VIDEO_PATH = ROOT / SPLIT / "videos" / f"{VIDEO_ID}.mp4"
ANN_PATH   = ROOT / SPLIT / "annotations" / f"{VIDEO_ID}.csv"
FRAMES_ROOT_HINT = ROOT / SPLIT / "frames"

print("SPLIT:", SPLIT)
print("VIDEO_ID:", VIDEO_ID)
print("VIDEO_PATH:", VIDEO_PATH)
print("ANN_PATH:", ANN_PATH)
print("FRAMES_ROOT_HINT:", FRAMES_ROOT_HINT)


In [ ]:
# ===== 2) Preset loading =====
if PRESET_PATH.exists():
    PRESETS = json.loads(PRESET_PATH.read_text(encoding="utf-8"))
    print(f"Loaded {len(PRESETS)} presets from {PRESET_PATH}")
else:
    raise FileNotFoundError(f"Preset file not found: {PRESET_PATH}")

if PRESET_NAME not in PRESETS:
    raise KeyError(f"Preset '{PRESET_NAME}' not found. Available: {list(PRESETS.keys())[:20]}")

CFG.update(PRESETS[PRESET_NAME])
print(f"Applied preset: {PRESET_NAME}")

ROOT = Path(CFG["root"])
TRAIN_FRAMES = ROOT / CFG["train_frames"]
VAL_FRAMES   = ROOT / CFG["val_frames"]
TRAIN_ANNS   = ROOT / CFG["train_anns"]
VAL_ANNS     = ROOT / CFG["val_anns"]

TRAIN_FLOW_FRAMES = Path(CFG.get("flow_frames_root_train", "flow/Training"))
VAL_FLOW_FRAMES   = Path(CFG.get("flow_frames_root_val",   "flow/Validation"))
TRAIN_SKELETON    = Path(CFG.get("skeleton_root_train", str(ROOT / "Training/skeleton")))
VAL_SKELETON      = Path(CFG.get("skeleton_root_val",   str(ROOT / "Validation/skeleton")))

USE_FLOW = bool(CFG.get("use_flow", False))
USE_SKELETON = bool(CFG.get("use_skeleton", False))

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")
print(f"USE_FLOW={USE_FLOW}, USE_SKELETON={USE_SKELETON}")
if USE_FLOW:
    print("TRAIN_FLOW_FRAMES:", TRAIN_FLOW_FRAMES)
    print("VAL_FLOW_FRAMES:", VAL_FLOW_FRAMES)
if USE_SKELETON:
    print("TRAIN_SKELETON:", TRAIN_SKELETON)
    print("VAL_SKELETON:", VAL_SKELETON)


In [ ]:
# ===== 3) Utils =====
def seed_everything(seed: int = 0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CFG["seed"])

def dummy_frame(size: int) -> np.ndarray:
    return np.zeros((size, size, 3), dtype=np.uint8)

def load_frame_labels(csv_path: Path) -> np.ndarray:
    df = pd.read_csv(csv_path)
    cols = {c.lower(): c for c in df.columns}
    fn_col = cols.get("frame_number", df.columns[0])
    lb_col = cols.get("label", df.columns[1])

    frame_num = df[fn_col].astype(int).to_numpy()
    label = df[lb_col].astype(int).to_numpy()

    n = int(frame_num.max())
    y = np.full((n,), -1, dtype=np.int64)
    idx = frame_num - 1
    ok = (idx >= 0) & (idx < n)
    y[idx[ok]] = label[ok]
    return y


In [ ]:
# ===== 4) Frame/video reading =====
def _read_clip_rgb_pyav(video_path: Path, start: int, clip_len: int, frame_step: int) -> Optional[np.ndarray]:
    if av is None:
        return None
    try:
        container = av.open(str(video_path))
        stream = next(s for s in container.streams if s.type == "video")
        fps = float(stream.average_rate) if stream.average_rate is not None else 30.0
        tb = float(stream.time_base)

        idxs = [int(start + k * frame_step) for k in range(clip_len)]
        want = set(idxs)
        first, last = idxs[0], idxs[-1]

        seek_time = max(0.0, first / fps)
        seek_pts = int(seek_time / tb) if tb > 0 else 0
        try:
            container.seek(seek_pts, stream=stream)
        except Exception:
            pass

        out = {}
        for frame in container.decode(stream):
            if frame.pts is None:
                continue
            fi = int(round(frame.pts * tb * fps))
            if fi < first - 2:
                continue
            if fi > last + 2:
                break
            if fi in want:
                out[fi] = frame.to_rgb().to_ndarray()
                if len(out) == len(want):
                    break

        container.close()
        if len(out) == 0:
            return None

        keys_sorted = sorted(out.keys())
        fallback = out[keys_sorted[0]]
        clip = []
        for fi in idxs:
            if fi in out:
                clip.append(out[fi])
            else:
                nearest = min(keys_sorted, key=lambda k: abs(k - fi))
                clip.append(out.get(nearest, fallback))
        return np.stack(clip, axis=0)
    except Exception:
        return None

def _read_clip_rgb_torchvision(video_path: Path, start: int, clip_len: int, frame_step: int) -> Optional[np.ndarray]:
    try:
        vr = torchvision.io.VideoReader(str(video_path), "video")
        meta = vr.get_metadata()
        fps = float(meta["video"]["fps"][0]) if "video" in meta and "fps" in meta["video"] else 30.0

        idxs = [int(start + k * frame_step) for k in range(clip_len)]
        out = []
        for fi in idxs:
            t = fi / fps
            vr.seek(t)
            fr = next(vr)
            x = fr["data"]
            if x.ndim == 3 and x.shape[-1] >= 3:
                x = x[..., :3]
            out.append(x.cpu().numpy())
        return np.stack(out, axis=0)
    except Exception:
        return None

def _read_frame_rgb(img_path: Path) -> Optional[np.ndarray]:
    try:
        x = torchvision.io.read_image(str(img_path))
        if x.ndim != 3:
            return None
        if x.shape[0] == 1:
            x = x.repeat(3, 1, 1)
        elif x.shape[0] >= 3:
            x = x[:3]
        return x.permute(1, 2, 0).cpu().numpy()
    except Exception:
        return None

def _build_frame_index(frames_dir: Path):
    paths = sorted(frames_dir.glob("*.jpg"))
    if len(paths) == 0:
        paths = sorted(frames_dir.glob("*.png"))
    if len(paths) == 0:
        return None

    nums = []
    keep = []
    for p in paths:
        m = None
        stem = p.stem
        mm = __import__("re").findall(r"(\d+)", stem)
        if mm:
            try:
                m = int(mm[-1])
            except Exception:
                m = None
        if m is not None:
            nums.append(m)
            keep.append(p)

    if len(keep) == 0:
        nums = list(range(1, len(paths) + 1))
        keep = paths

    return np.asarray(nums, dtype=np.int64), keep

def _read_clip_rgb_frames(frames_dir: Path, start: int, clip_len: int, frame_step: int, frame_index=None) -> Optional[np.ndarray]:
    if frame_index is None:
        frame_index = _build_frame_index(frames_dir)
    if frame_index is None:
        return None

    nums, paths = frame_index
    idxs = [int(start + k * frame_step + 1) for k in range(clip_len)]  # annotations are 1-based
    clip = []
    for fi in idxs:
        pos = np.searchsorted(nums, fi)
        cand = []
        if pos < len(nums):
            cand.append(pos)
        if pos > 0:
            cand.append(pos - 1)
        if len(cand) == 0:
            return None
        best = min(cand, key=lambda j: abs(int(nums[j]) - fi))
        fr = _read_frame_rgb(paths[best])
        if fr is None:
            return None
        clip.append(fr)
    return np.stack(clip, axis=0)

def read_clip_rgb(video_path: Optional[Path], start: int, clip_len: int, frame_step: int,
                  backend: str = "pyav", frames_dir: Optional[Path] = None, frame_index=None) -> Optional[np.ndarray]:
    if backend == "pyav":
        return _read_clip_rgb_pyav(video_path, start, clip_len, frame_step)
    if backend == "torchvision":
        return _read_clip_rgb_torchvision(video_path, start, clip_len, frame_step)
    if backend == "frames":
        if frames_dir is None:
            raise ValueError("backend='frames' requires frames_dir")
        return _read_clip_rgb_frames(frames_dir, start, clip_len, frame_step, frame_index=frame_index)
    raise ValueError(f"Unknown backend: {backend}")


In [ ]:
# ===== 5) Transforms =====
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

class TorchVideoTransform:
    def __init__(self, size: int, train: bool):
        self.size = int(size)
        self.train = bool(train)

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        if not isinstance(x, torch.Tensor):
            x = torch.from_numpy(x)

        if x.ndim == 3 and x.shape[-1] in (1, 3, 4) and x.shape[0] not in (1, 3, 4):
            x = x.permute(2, 0, 1)

        if x.shape[0] == 4:
            x = x[:3]

        if x.dtype == torch.uint8:
            x = x.float().div_(255.0)
        else:
            x = x.float()
            if x.max() > 1.5:
                x = x / 255.0

        x = TF.resize(x, [self.size, self.size], antialias=True)
        x = TF.normalize(x, mean=IMAGENET_MEAN, std=IMAGENET_STD)
        return x

val_tfms = TorchVideoTransform(CFG["img_size"], train=False)


In [ ]:
# ===== 6) Model build + checkpoint loading =====
import sys
_models_dir = str(Path.cwd())
if _models_dir not in sys.path:
    sys.path.insert(0, _models_dir)

from models import build_model, ALL_BACKBONES

print(f"Loaded models.py — {len(ALL_BACKBONES)} backbones available")

backbone = CFG["backbone"]
print(f"Building model: {backbone}")

model = build_model(
    backbone=backbone,
    num_classes=2,
    pretrained=True,
    dropout=float(CFG.get("dropout", 0.3)),
    cfg=CFG,
).to(device)

def _resolve_ckpt_path(run_dir: Optional[Path], ckpt_path: Optional[Path]) -> Path:
    if ckpt_path is not None:
        return Path(ckpt_path)
    if run_dir is None:
        raise ValueError("Provide CKPT_PATH or RUN_DIR")
    run_dir = Path(run_dir)
    for name in ["best.pt", "best.pth", "checkpoint_best.pt", "last.pt"]:
        p = run_dir / name
        if p.exists():
            return p
    pts = sorted(run_dir.glob("*.pt"))
    if pts:
        return pts[0]
    raise FileNotFoundError(f"No checkpoint found in {run_dir}")

def load_checkpoint_flexible(model: nn.Module, ckpt_path: Path, device: str = "cuda"):
    ckpt = torch.load(str(ckpt_path), map_location=device)
    state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"Loaded checkpoint: {ckpt_path}")
    print(f"Missing keys: {len(missing)}, Unexpected keys: {len(unexpected)}")
    return ckpt, missing, unexpected

if CKPT_PATH is None and RUN_DIR is not None:
    CKPT_PATH = _resolve_ckpt_path(Path(RUN_DIR), None)
elif CKPT_PATH is not None:
    CKPT_PATH = Path(CKPT_PATH)

if CKPT_PATH is not None:
    _ckpt, _missing, _unexpected = load_checkpoint_flexible(model, CKPT_PATH, device=device)
else:
    print("[warn] checkpoint not loaded yet — set CKPT_PATH or RUN_DIR above")

model.eval();


In [ ]:
# ===== 7) Inference function =====
@torch.no_grad()
def infer_video_per_frame(
    model: nn.Module,
    video_path: Optional[Path] = None,
    video_id: Optional[str] = None,
    frames_root: Optional[Path] = None,
    ann_path: Optional[Path] = None,
    clip_len: int = 16,
    frame_step: int = 1,
    stride: Optional[int] = None,
    threshold: float = 0.5,
    transform=None,
    backend: str = "pyav",
    device: str = "cuda",
    debug: bool = True,
    flow_frames_root: Optional[Path] = None,
):
    model.eval()
    if transform is None:
        transform = val_tfms
    if stride is None:
        stride = max(1, (clip_len * frame_step) // 2)

    n_frames = 0
    if ann_path is not None and Path(ann_path).exists():
        out = load_frame_labels(Path(ann_path))
        labels = out[0] if isinstance(out, tuple) else out
        n_frames = int(len(labels))

    frame_index = None
    frames_dir = None
    if backend == "frames":
        if video_id is None:
            if video_path is None:
                raise ValueError("For backend='frames' provide video_id or video_path")
            video_id = Path(video_path).stem
        if frames_root is None:
            raise ValueError("For backend='frames' provide frames_root")
        frames_dir = Path(frames_root) / str(video_id)
        frame_index = _build_frame_index(frames_dir)
        if n_frames <= 0 and frame_index is not None and len(frame_index[0]) > 0:
            n_frames = int(frame_index[0][-1])

    if n_frames <= 0 and backend != "frames" and video_path is not None:
        try:
            import cv2
            cap = cv2.VideoCapture(str(video_path))
            if cap.isOpened():
                n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            cap.release()
        except Exception:
            pass

    if n_frames <= 0 and backend != "frames" and video_path is not None and av is not None:
        try:
            container = av.open(str(video_path))
            stream = next(s for s in container.streams if s.type == "video")
            n_frames = int(stream.frames) if stream.frames else 0
            if n_frames <= 0 and stream.duration and stream.time_base and stream.average_rate:
                n_frames = int(float(stream.duration * stream.time_base) * float(stream.average_rate))
            container.close()
        except Exception:
            n_frames = 0

    if debug:
        src = str(frames_dir) if backend == "frames" else (video_path.name if video_path is not None else "None")
        print(f"[infer] src={src} n_frames={n_frames} backend={backend} frame_step={frame_step} stride={stride}")

    if n_frames <= 0:
        return np.zeros((0,), np.float32), np.zeros((0,), np.int64)

    prob_sum = np.zeros((n_frames,), dtype=np.float32)
    prob_cnt = np.zeros((n_frames,), dtype=np.float32)

    span = (clip_len - 1) * frame_step + 1
    last_start = max(0, n_frames - span)
    starts = list(range(0, last_start + 1, stride))
    if len(starts) == 0:
        starts = [0]
    if starts[-1] != last_start:
        starts.append(last_start)

    for start in tqdm(starts, desc="infer", dynamic_ncols=True, disable=not debug):
        if backend == "frames":
            rgb = read_clip_rgb(
                video_path=None,
                start=start,
                clip_len=clip_len,
                frame_step=frame_step,
                backend=backend,
                frames_dir=frames_dir,
                frame_index=frame_index,
            )
        else:
            rgb = read_clip_rgb(
                video_path=video_path,
                start=start,
                clip_len=clip_len,
                frame_step=frame_step,
                backend=backend,
            )

        if rgb is None or rgb.shape[0] != clip_len:
            rgb = np.stack([dummy_frame(CFG["img_size"])] * clip_len, axis=0)

        frames = []
        for t in range(clip_len):
            fr_t = torch.from_numpy(rgb[t]).permute(2, 0, 1)
            frames.append(transform(fr_t))
        clip = torch.stack(frames, dim=0).permute(1, 0, 2, 3).unsqueeze(0).to(device)

        kwargs = {}
        if flow_frames_root is not None:
            flow_dir = Path(flow_frames_root) / str(video_id)
            flow_fi = _build_frame_index(flow_dir) if flow_dir.exists() else None
            flow_rgb = read_clip_rgb(
                video_path=None, start=start, clip_len=clip_len,
                frame_step=frame_step, backend="frames",
                frames_dir=flow_dir, frame_index=flow_fi,
            )
            if flow_rgb is None or flow_rgb.shape[0] != clip_len:
                flow_rgb = np.stack([dummy_frame(CFG["img_size"])] * clip_len, axis=0)
            flow_frames_list = []
            for t in range(clip_len):
                fr_t = torch.from_numpy(flow_rgb[t]).permute(2, 0, 1)
                flow_frames_list.append(transform(fr_t))
            kwargs["flow"] = torch.stack(flow_frames_list, dim=0).permute(1, 0, 2, 3).unsqueeze(0).to(device)

        if USE_SKELETON:
            sk_root = TRAIN_SKELETON if (frames_root is not None and str(frames_root).endswith("Training/frames")) else VAL_SKELETON
            sk_path = Path(sk_root) / f"{video_id}.npy"
            if sk_path.exists():
                sk = np.load(sk_path)
                idxs = start + np.arange(clip_len) * frame_step
                idxs = np.clip(idxs, 0, max(len(sk)-1, 0)).astype(int)
                sk_clip = torch.from_numpy(sk[idxs]).float().unsqueeze(0).to(device)   # [1,T,D]
                kwargs["skeleton"] = sk_clip
            else:
                print(f"[warn] skeleton file not found: {sk_path}")

        logits = model(clip, **kwargs) if kwargs else model(clip)
        probs1 = torch.softmax(logits, dim=1)[:, 1, :].squeeze(0).detach().cpu().numpy()

        idxs = start + np.arange(clip_len) * frame_step
        idxs = np.clip(idxs, 0, n_frames - 1).astype(int)
        for k, fi in enumerate(idxs):
            prob_sum[fi] += float(probs1[k])
            prob_cnt[fi] += 1.0

    probs = prob_sum / np.maximum(prob_cnt, 1.0)
    pred = (probs >= threshold).astype(np.int64)

    if debug:
        covered = int((prob_cnt > 0).sum())
        print(f"[infer] covered_frames={covered}/{n_frames} ({covered/n_frames:.1%}) probs[min,max]=({probs.min():.3f},{probs.max():.3f})")

    return probs, pred

def smooth_probs(probs: np.ndarray, win: int = 7) -> np.ndarray:
    if win <= 1 or probs.size == 0:
        return probs
    pad = win // 2
    x = np.pad(probs, (pad, pad), mode="edge")
    kernel = np.ones((win,), dtype=np.float32) / win
    return np.convolve(x, kernel, mode="valid")


In [ ]:
# ===== 8) Helper: resolve video/annotation =====
def resolve_video_and_ann(video_id=None, video_path=None, ann_path=None, backend="frames", split=None):
    video_id_out = video_id
    video_path_out = Path(video_path) if video_path is not None else None
    ann_path_out = Path(ann_path) if ann_path is not None else None

    if video_id_out is None and video_path_out is not None:
        video_id_out = video_path_out.stem

    split = str(split) if split is not None else None
    split_frames = None
    split_anns = None
    split_flow = None
    if split == "Validation":
        split_frames = VAL_FRAMES
        split_anns = VAL_ANNS
        split_flow = VAL_FLOW_FRAMES if USE_FLOW else None
    elif split == "Training":
        split_frames = TRAIN_FRAMES
        split_anns = TRAIN_ANNS
        split_flow = TRAIN_FLOW_FRAMES if USE_FLOW else None

    if ann_path_out is None and video_id_out is not None:
        # Prefer the user-selected split first
        if split_anns is not None:
            cand = split_anns / f"{video_id_out}.csv"
            if cand.exists():
                ann_path_out = cand

        # Fallback to auto-discovery
        if ann_path_out is None:
            cand_val = VAL_ANNS / f"{video_id_out}.csv"
            cand_tr  = TRAIN_ANNS / f"{video_id_out}.csv"
            if cand_val.exists():
                ann_path_out = cand_val
            elif cand_tr.exists():
                ann_path_out = cand_tr

    if backend == "frames":
        if video_id_out is None:
            raise ValueError("For backend='frames', set VIDEO_ID")

        # Prefer the user-selected split first
        if split_frames is not None and (split_frames / video_id_out).exists():
            frames_root = split_frames
            flow_root = split_flow
        elif (VAL_FRAMES / video_id_out).exists():
            frames_root = VAL_FRAMES
            flow_root = VAL_FLOW_FRAMES if USE_FLOW else None
        elif (TRAIN_FRAMES / video_id_out).exists():
            frames_root = TRAIN_FRAMES
            flow_root = TRAIN_FLOW_FRAMES if USE_FLOW else None
        else:
            raise FileNotFoundError(f"Could not find frame folder for video_id={video_id_out} in train/val frames")

        return video_id_out, None, ann_path_out, frames_root, flow_root
    else:
        if video_path_out is None:
            raise ValueError("For backend!='frames', provide VIDEO_PATH")
        flow_root = None
        return video_id_out, video_path_out, ann_path_out, None, flow_root

video_id_res, video_path_res, ann_path_res, frames_root_res, flow_root_res = resolve_video_and_ann(
    video_id=VIDEO_ID,
    video_path=VIDEO_PATH,
    ann_path=ANN_PATH,
    backend=CFG["backend"],
    split=SPLIT,
)

print("video_id:", video_id_res)
print("video_path:", video_path_res)
print("ann_path:", ann_path_res)
print("frames_root:", frames_root_res)
print("flow_root:", flow_root_res)


In [ ]:
# ===== 9) Run inference =====
SMOOTH_WIN = 1
THRESHOLD = 0.5
STRIDE = None

probs_raw, pred_raw = infer_video_per_frame(
    model=model,
    video_path=video_path_res,
    video_id=video_id_res,
    frames_root=frames_root_res,
    ann_path=ann_path_res,
    clip_len=int(CFG["clip_len"]),
    frame_step=int(CFG["frame_step"]),
    stride=STRIDE,
    threshold=THRESHOLD,
    transform=val_tfms,
    backend=CFG["backend"],
    device=device,
    debug=True,
    flow_frames_root=flow_root_res,
)

probs = smooth_probs(probs_raw, win=int(SMOOTH_WIN))
pred = (probs >= THRESHOLD).astype(np.int64)

y_true = None
if ann_path_res is not None and Path(ann_path_res).exists():
    out = load_frame_labels(Path(ann_path_res))
    y_true = out[0] if isinstance(out, tuple) else out
    y_true = np.asarray(y_true).reshape(-1)

    L = min(len(y_true), len(probs))
    y_true = y_true[:L]
    probs = probs[:L]
    pred = pred[:L]

    m_labeled = (y_true != -1)
    n_labeled = int(m_labeled.sum())
    n_pos = int(((y_true == 1) & m_labeled).sum())
    n_neg = int(((y_true == 0) & m_labeled).sum())
    print(f"[GT] loaded: {ann_path_res}")
    print(f"[GT] labeled={n_labeled}/{L}, pos={n_pos}, neg={n_neg}")
else:
    L = len(probs)
    print(f"[GT] annotation is missing for this video: {ann_path_res}")

df_plot = pd.DataFrame({
    "frame_idx": np.arange(L),
    "prob_violent": probs,
    "pred": pred,
})
if y_true is not None:
    # Keep both names for compatibility with old cells.
    df_plot["y_true"] = y_true
    df_plot["true"] = y_true

display(df_plot.head())
print("n_frames:", len(df_plot))

In [ ]:
# ===== 10) Plot probabilities + GT/pred segments (clean) =====
def _segments_from_binary(x, y):
    """
    x: frame indices, shape [N]
    y: binary {0,1}, shape [N]
    returns list of (start, end) inclusive-ish ranges in x coordinates
    """
    x = np.asarray(x)
    y = np.asarray(y).astype(int)
    segs = []
    if len(x) == 0:
        return segs
    in_seg = False
    start = None
    prev_x = None
    for xi, yi in zip(x, y):
        if yi == 1 and not in_seg:
            in_seg = True
            start = xi
        if yi == 0 and in_seg:
            segs.append((start, prev_x))
            in_seg = False
            start = None
        prev_x = xi
    if in_seg:
        segs.append((start, prev_x))
    return segs


def plot_video_probs_simple(df_plot, threshold=0.5, start=None, end=None, show_pred=True):
    import matplotlib.pyplot as plt
    import numpy as np

    df = df_plot.copy()
    if start is not None or end is not None:
        df = df.iloc[start:end].reset_index(drop=True)

    x = np.arange(len(df))

    # Support both naming schemes:
    # old: prob/true ; current notebook: prob_violent/y_true
    prob_col = "prob" if "prob" in df.columns else "prob_violent"
    true_col = "y_true" if "y_true" in df.columns else ("true" if "true" in df.columns else None)

    prob = df[prob_col].to_numpy()

    plt.figure(figsize=(18, 5))

    # probability as transparent points
    plt.scatter(x, prob, s=8, alpha=0.35, label="P(violent)")

    # true labels (if available): explicit step + positive spans
    if true_col is not None:
        true = df[true_col].astype(int).to_numpy()
        m = true != -1
        if m.any():
            true_vis = np.where(m, true, np.nan)
            plt.step(x, true_vis, where="post", linewidth=1.2, alpha=0.9, label="true 0/1")

            true_bin = np.where(m, true, 0)
            for l, r in _segments_from_binary(x, true_bin):
                plt.axvspan(l, r + 1, color="tab:green", alpha=0.10)
        else:
            plt.text(0.01, 0.95, "GT present but all labels are -1", transform=plt.gca().transAxes,
                     fontsize=10, color="tab:red", va="top")
    else:
        plt.text(0.01, 0.95, "GT column is missing", transform=plt.gca().transAxes,
                 fontsize=10, color="tab:red", va="top")

    # optional predicted labels
    if show_pred and "pred" in df.columns:
        pred = df["pred"].astype(int).to_numpy()
        plt.step(x, pred, where="post", linewidth=1.0, alpha=0.45, label="pred 0/1")

    plt.axhline(threshold, linestyle="--", linewidth=1, label=f"threshold={threshold:.2f}")

    plt.ylim(-0.08, 1.08)
    plt.xlabel("Frame index")
    plt.ylabel("Probability / label")
    plt.title("Video prediction overview")
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()


plot_video_probs_simple(df_plot, threshold=THRESHOLD, show_pred=False)

In [ ]:
# ===== 12) Metrics + export =====
if "y_true" in df_plot.columns:
    mask = df_plot["y_true"].to_numpy() != -1
    y_true_eval = df_plot.loc[mask, "y_true"].to_numpy().astype(int)
    y_pred_eval = df_plot.loc[mask, "pred"].to_numpy().astype(int)

    from sklearn.metrics import f1_score, classification_report
    print("macro_f1:", f1_score(y_true_eval, y_pred_eval, average="macro"))
    print(classification_report(y_true_eval, y_pred_eval, digits=4))
else:
    print("No y_true available.")

if EXPORT_CSV:
    out_csv = EXPORT_DIR / f"{video_id_res or 'video'}__{PRESET_NAME}.csv"
    df_plot.to_csv(out_csv, index=False)
    print("Saved:", out_csv)
